# IntelSight Workshop Lab

This notebook is the workshop walkthrough for every IntelSight module and the
integrations between them. Run the cells top to bottom; each module section is
self-contained, shows real pipeline artifacts, and exports its visualization to
`output/lab-artifacts/` for the web-dashboard and Tauri desktop app to display.

## Workshop structure

| Section | Module | What you learn |
|---|---|---|
| Module Map | integration | data-flow diagram between all modules |
| Module 1 | `flightrecord-parser` (Rust) | TXT flight log -> telemetry CSV/GeoJSON |
| Module 2 | `flight-visualizer` | SRT telemetry parsing and fields |
| Module 3 | `cv-pipeline` | detection, plate candidates, OCR, tracking |
| Module 4 | `cv-pipeline` sync + fuse | geotagging, multi-frame fusion, report |
| Module 5 | `render_overlay_video` | optical-flow motion view for the overlay |
| Lab export | integration | manifest of demo PNGs consumed by the apps |

## Guided exercises

- Module 1: compare `latitude/longitude` from the TXT frames CSV against the SRT CSV for the same mission; estimate the telemetry-rate difference.
- Module 2: find the SRT rows where `focal_len` changes; explain why the Mini 4 Pro reports 35mm-equivalent values.
- Module 3: change `FRAME_PAIR_INDEX` and re-run detection; observe how box counts change with scene density.
- Module 4: look at `geolocation_mode` and `geo_spread_m` in the fused output; find one `proxy` observation and one `telemetry` observation.
- Module 5: compare the flow overlay between a static scene window and a moving-scene window.



## Pipeline Outline

The current production path is:
- sample frames from mission video
- detect vehicles with YOLO segmentation
- propose plate regions, optionally detect full-frame plate boxes
- OCR only selected candidate crops
- reuse OCR when frame difference and re-identification show the object is stable
- sync detections with SRT telemetry
- fuse repeated observations into a geospatial object record
- export HTML, GeoJSON, overlay video, and dashboard/database artifacts

## Overlay Color Semantics

The overlay renderer currently uses these colors:

- yellow outlines and translucent fill for `vehicle_boxes`
- green outlines and translucent fill for `ocr` boxes
- green does **not** automatically mean a true resolved license plate
- a green box can still be a weak or incorrect plate candidate if `ocr_text` is empty and the box geometry is too large

The next sections make that visible using real JSONL output from the existing pipeline so the notebook can serve as documentation instead of relying on terminal logs.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'cv-pipeline' else Path.cwd().resolve()
MODULE_DIR = ROOT / 'modules' / 'cv-pipeline'
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

import run_cv_pipeline as rcp
import license_plate_service as lps
import build_detection_report as bdr
import render_overlay_video as rov

plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['axes.grid'] = False

In [ ]:
MISSION_DIR = ROOT / 'data' / 'flightrecords' / 'flight_mission_drone' / 'FlagerPublix'
OUTPUT_DIR = ROOT / 'output' / 'cv' / 'FlagerPublix'
video_paths = sorted(MISSION_DIR.glob('*.MP4'))
if not video_paths:
    raise FileNotFoundError(f'No MP4 files found in {MISSION_DIR}')

VIDEO_PATH = video_paths[0]
FRAME_STEP = 2
FRAME_PAIR_INDEX = 240
DEVICE = '0'

print('Video:', VIDEO_PATH.name)
print('Output dir:', OUTPUT_DIR)
print('Metadata:', rcp.video_metadata(VIDEO_PATH))

In [ ]:
LAB_DIR = ROOT / 'output' / 'lab-artifacts'
LAB_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib.patches as mpatches


def draw_pipeline_diagram(out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(17, 6))
    ax.axis('off')
    stage_rows = [
        ('1 Ingest', ['MP4 video', 'TXT flight log', 'SRT telemetry'], '#dbeafe'),
        ('2 Parse', ['flightrecord-parser\n(Rust)', 'flight-visualizer\nSRT parser'], '#dcfce7'),
        ('3 Perceive', ['cv-pipeline\ndetect + OCR', 'plate service\ncandidates + fusion'], '#fef9c3'),
        ('4 Ground', ['SRT sync\ngeotagging', 'fuse\nmulti-frame'], '#ffedd5'),
        ('5 Report', ['lp_vehicle_report\nHTML / GeoJSON', 'overlay video\nmotion viz'], '#fce7f3'),
        ('6 Review', ['web-dashboard\nserver review', 'desktop-app\nTauri UX', 'PostGIS + API\n(planned)'], '#e0e7ff'),
    ]
    col_w = 2.2
    row_h = 1.0
    for col, (title, boxes, color) in enumerate(stage_rows):
        x = 0.4 + col * (col_w + 0.22)
        ax.text(x + col_w / 2, 2.95, title, ha='center', va='center', fontsize=12, fontweight='bold')
        for i, label in enumerate(boxes):
            y = 1.7 - i * (row_h + 0.12)
            rect = mpatches.FancyBboxPatch(
                (x, y - 0.42), col_w, 0.85,
                boxstyle='round,pad=0.08', linewidth=1.2,
                edgecolor='#475569', facecolor=color)
            ax.add_patch(rect)
            ax.text(x + col_w / 2, y, label, ha='center', va='center', fontsize=9)
        if col < len(stage_rows) - 1:
            ax.annotate('', xy=(x + col_w + 0.2, 1.7), xytext=(x + col_w, 1.7),
                        arrowprops=dict(arrowstyle='-|>', color='#334155', lw=2))
    ax.set_xlim(0, len(stage_rows) * (col_w + 0.22) + 0.3)
    ax.set_ylim(-0.6, 3.3)
    fig.suptitle('IntelSight module integration data flow', fontsize=14)
    fig.savefig(out_path, dpi=130, bbox_inches='tight')
    plt.close(fig)
    print('saved', out_path)


draw_pipeline_diagram(LAB_DIR / 'pipeline_dataflow.png')



## Module 1 · flightrecord-parser (Rust)

The Rust module in `modules/flightrecord-parser/` converts DJI TXT flight logs
into normalized telemetry. The exported `frames.csv` is the ground truth for
IMU attitude (`pitch/roll/yaw`) that the SRT files do not carry.

The cell below loads a real `frames.csv` from `output/flightrecords/` and plots
the ENU trajectory, altitude profile, and heading distribution. This is the
data the future camera-projection and SfM stages will anchor to.



In [ ]:
import glob as _glob

frames_files = sorted(ROOT.glob('output/flightrecords/*.frames.csv'))
if not frames_files:
    print('No frames.csv found under output/flightrecords. Run scripts/run_flightrecord_pipeline.sh first.')
else:
    frames_path = frames_files[0]
    print('Demo file:', frames_path.name)
    telemetry = pd.read_csv(frames_path)
    print('Columns:', list(telemetry.columns))
    print(telemetry[['customDateTime', 'latitude', 'longitude', 'height', 'pitch', 'roll', 'yaw']].head(3).to_string(index=False))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(telemetry['longitude'], telemetry['latitude'], marker='.', markersize=2, linewidth=1)
    axes[0].set_title('Trajectory (lon/lat)')
    axes[0].set_xlabel('longitude'); axes[0].set_ylabel('latitude')
    axes[1].plot(telemetry['height'], marker='.', markersize=2, linewidth=1)
    axes[1].set_title('Barometric height over samples')
    axes[1].set_xlabel('sample index'); axes[1].set_ylabel('height (m)')
    axes[2].hist(telemetry['yaw'].dropna(), bins=36)
    axes[2].set_title('Aircraft yaw distribution')
    axes[2].set_xlabel('yaw (deg)')
    for ax in axes:
        ax.grid(True, alpha=0.25)
    fig.suptitle(f'Module 1: Rust parser output — {frames_path.stem[:60]}')
    fig.tight_layout()
    fig.savefig(LAB_DIR / 'module1_parser_telemetry.png', dpi=130)
    plt.show()



## Module 2 · flight-visualizer (SRT telemetry)

`modules/flight-visualizer/parse_dji_srt.py` extracts per-frame telemetry from
the DJI SRT subtitle stream: pose, altitude, camera parameters, and
`focal_len` (35mm-equivalent). The SRT is the time-aligned telemetry source
for frame geotagging.



In [ ]:
srt_files = sorted(ROOT.glob('output/flightrecords/flight_mission_drone/FlagerPublix/*.srt.csv'))
if not srt_files:
    srt_files = sorted(ROOT.glob('output/flightrecords/flight_mission_drone/*.srt.csv'))
if not srt_files:
    print('No SRT CSVs found. Run scripts/run_srt_dashboard.sh first.')
else:
    srt_path = srt_files[0]
    print('Demo file:', srt_path.name)
    srt = pd.read_csv(srt_path)
    print('Columns:', list(srt.columns))
    srt['t_sec'] = (srt['diff_ms'] - srt['diff_ms'].iloc[0]) / 1000.0

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(srt['t_sec'], srt['rel_alt'], linewidth=1)
    axes[0].set_title('Relative altitude over mission time')
    axes[0].set_xlabel('time (s)'); axes[0].set_ylabel('rel_alt (m)')
    axes[1].plot(srt['t_sec'], srt['focal_len'], linewidth=1, color='tab:orange')
    axes[1].set_title('Focal length (35mm equiv) over time')
    axes[1].set_xlabel('time (s)'); axes[1].set_ylabel('focal_len (mm)')
    axes[2].plot(srt['longitude'], srt['latitude'], marker='.', markersize=2, linewidth=1, color='tab:green')
    axes[2].set_title('SRT trajectory (lon/lat)')
    axes[2].set_xlabel('longitude'); axes[2].set_ylabel('latitude')
    for ax in axes:
        ax.grid(True, alpha=0.25)
    fig.suptitle(f'Module 2: SRT telemetry — {srt_path.stem[:60]}')
    fig.tight_layout()
    fig.savefig(LAB_DIR / 'module2_srt_telemetry.png', dpi=130)
    plt.show()



## Module 3 · cv-pipeline (detection, plates, OCR, tracking)

The next cells (already part of this notebook) walk through the perception
module:

- `LazyModels` lazy-loads the segmentation vehicle model and the plate model.
- `license_plate_service` proposes plate candidates from vehicle crops.
- `run_plate_ocr` gates OCR by sharpness and box geometry.
- `assign_vehicle_tracks` links detections across frames using IoU +
  signature similarity, which drives OCR reuse decisions.

Continue to the cells below to inspect each stage visually.



In [ ]:
def read_frame(video_path: Path, frame_idx: int) -> np.ndarray:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Unable to open {video_path}')
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f'Unable to read frame {frame_idx} from {video_path.name}')
    return frame

def as_rgb(frame: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

def draw_boxes(frame: np.ndarray, boxes: list[dict], color: tuple[int, int, int], label_key: str | None = None) -> np.ndarray:
    canvas = frame.copy()
    for item in boxes:
        x1, y1, x2, y2 = [int(v) for v in item['xyxy']]
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 2)
        if label_key is not None and item.get(label_key):
            cv2.putText(canvas, str(item[label_key]), (x1, max(18, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    return canvas

def show_side_by_side(left: np.ndarray, right: np.ndarray, left_title: str, right_title: str):
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    axes[0].imshow(as_rgb(left))
    axes[0].set_title(left_title)
    axes[0].axis('off')
    axes[1].imshow(as_rgb(right))
    axes[1].set_title(right_title)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
def render_optical_flow(prev_frame: np.ndarray, curr_frame: np.ndarray, box: list[int] | tuple[int, int, int, int] | None = None, motion_scale: float = 30.0, alpha: float = 0.55, vector_step: int = 12) -> tuple[np.ndarray, dict]:
    """Render a Farneback optical-flow overlay for a frame pair and optional vehicle ROI."""
    if prev_frame is None or curr_frame is None:
        raise ValueError('Both frames must be valid OpenCV images')

    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)

    if box is not None:
        x1, y1, x2, y2 = [int(v) for v in box]
        x1 = max(0, min(prev_frame.shape[1] - 1, x1))
        y1 = max(0, min(prev_frame.shape[0] - 1, y1))
        x2 = max(x1 + 1, min(prev_frame.shape[1], x2))
        y2 = max(y1 + 1, min(prev_frame.shape[0], y2))
        prev_roi = prev_gray[y1:y2, x1:x2]
        curr_roi = curr_gray[y1:y2, x1:x2]
    else:
        prev_roi = prev_gray
        curr_roi = curr_gray

    if prev_roi.size == 0 or curr_roi.size == 0:
        raise ValueError('The motion ROI is empty after cropping')

    flow = cv2.calcOpticalFlowFarneback(
        prev_roi,
        curr_roi,
        None,
        pyr_scale=0.5,
        levels=3,
        winsize=15,
        iterations=3,
        poly_n=5,
        poly_sigma=1.2,
        flags=cv2.OPTFLOW_FARNEBACK_GAUSSIAN,
    )
    flow_x = flow[..., 0]
    flow_y = flow[..., 1]
    motion_mag, _ = cv2.cartToPolar(flow_x, flow_y)

    normalized = np.clip((motion_mag / max(1e-6, motion_scale)) * 255.0, 0, 255).astype(np.uint8)
    heatmap = cv2.applyColorMap(normalized, cv2.COLORMAP_TURBO)

    overlay = curr_frame.copy()
    roi_x1 = x1 if box is not None else 0
    roi_y1 = y1 if box is not None else 0
    roi_x2 = x2 if box is not None else overlay.shape[1]
    roi_y2 = y2 if box is not None else overlay.shape[0]

    if box is not None:
        roi = overlay[roi_y1:roi_y2, roi_x1:roi_x2]
        blending = cv2.addWeighted(roi, 1.0 - alpha, heatmap, alpha, 0)
        overlay[roi_y1:roi_y2, roi_x1:roi_x2] = blending
        cv2.rectangle(overlay, (roi_x1, roi_y1), (roi_x2 - 1, roi_y2 - 1), (0, 255, 255), 2)
    else:
        overlay = cv2.addWeighted(overlay, 1.0 - alpha, heatmap, alpha, 0)

    for yy in range(0, flow_x.shape[0], max(1, vector_step)):
        for xx in range(0, flow_x.shape[1], max(1, vector_step)):
            dx = float(flow_x[yy, xx])
            dy = float(flow_y[yy, xx])
            speed = float(np.hypot(dx, dy))
            if speed < 0.45:
                continue
            start_x = xx + (roi_x1 if box is not None else 0)
            start_y = yy + (roi_y1 if box is not None else 0)
            end_x = int(start_x + dx * 2.4)
            end_y = int(start_y + dy * 2.4)
            end_x = max(0, min(overlay.shape[1] - 1, end_x))
            end_y = max(0, min(overlay.shape[0] - 1, end_y))
            color = (0, 255, 0) if speed > 2.0 else (0, 200, 255)
            cv2.arrowedLine(overlay, (start_x, start_y), (end_x, end_y), color, 2, cv2.LINE_AA, tipLength=0.35)
            cv2.circle(overlay, (start_x, start_y), 2, color, -1)

    summary = {
        'mean_motion_px': float(np.mean(motion_mag)),
        'max_motion_px': float(np.max(motion_mag)),
        'active_ratio': float(np.mean(motion_mag > (motion_scale * 0.15))),
        'box': [roi_x1, roi_y1, roi_x2, roi_y2] if box is not None else None,
    }
    return overlay, summary


In [ ]:
def choose_motion_roi(frame_shape: tuple[int, int], record: dict | None = None, fallback_roi: tuple[int, int, int, int] | None = None, padding: int = 30) -> tuple[int, int, int, int]:
    """Prefer a vehicle-centered ROI, then fall back to a static landmark or full frame."""
    if record is not None:
        vehicles = record.get('vehicle_boxes', []) or []
        if vehicles:
            best = max(vehicles, key=lambda item: float(item.get('conf', 0.0)) * max(1.0, float(item.get('area', 1.0))))
            xyxy = best.get('xyxy', [0, 0, 0, 0])
            x1, y1, x2, y2 = [int(v) for v in xyxy]
            x1 = max(0, x1 - padding)
            y1 = max(0, y1 - padding)
            x2 = min(frame_shape[1], x2 + padding)
            y2 = min(frame_shape[0], y2 + padding)
            return (x1, y1, x2, y2)
    if fallback_roi is not None:
        return fallback_roi
    return (0, 0, frame_shape[1], frame_shape[0])


def summarize_object_stats(frame_indices: list[int], records_by_frame: dict[int, dict]) -> list[dict]:
    rows = []
    for idx in frame_indices:
        record = records_by_frame.get(int(idx))
        if record is None:
            rows.append({
                'frame': idx,
                'vehicle_count': 0,
                'mean_conf': 0.0,
                'mean_width': 0.0,
                'mean_height': 0.0,
            })
            continue
        vehicles = record.get('vehicle_boxes', []) or []
        widths = []
        heights = []
        confs = []
        for vehicle in vehicles:
            x1, y1, x2, y2 = [int(v) for v in vehicle.get('xyxy', [0, 0, 0, 0])]
            widths.append(max(0, x2 - x1))
            heights.append(max(0, y2 - y1))
            confs.append(float(vehicle.get('conf', 0.0)))
        rows.append({
            'frame': idx,
            'vehicle_count': len(vehicles),
            'mean_conf': float(np.mean(confs)) if confs else 0.0,
            'mean_width': float(np.mean(widths)) if widths else 0.0,
            'mean_height': float(np.mean(heights)) if heights else 0.0,
        })
    return rows


def summarize_motion_history(frame_indices: list[int], roi: tuple[int, int, int, int] | None = None, motion_scale: float = 35.0) -> list[dict]:
    """Compute per-step translation, rotation, and 3D-style motion proxies across a short frame sequence."""
    traces = []
    for idx in range(1, len(frame_indices)):
        prev_idx = frame_indices[idx - 1]
        curr_idx = frame_indices[idx]
        prev_frame = read_frame(VIDEO_PATH, prev_idx)
        curr_frame = read_frame(VIDEO_PATH, curr_idx)
        if prev_frame is None or curr_frame is None:
            continue
        metrics = rov.estimate_optical_flow_kinematics(
            prev_frame,
            curr_frame,
            roi=roi,
            motion_scale=motion_scale,
            vector_step=10,
            altitude_m=35.0,
        )
        traces.append({
            'frame_a': prev_idx,
            'frame_b': curr_idx,
            'translation_px': metrics['translation_px'],
            'translation_m': metrics.get('translation_m', 0.0),
            'rotation_proxy_deg': metrics['rotation_proxy_deg'],
            'mean_speed_px': metrics['mean_speed_px'],
            'translation_vector_px': metrics['translation_vector_px'],
            'translation_vector_m': metrics.get('translation_vector_m', (0.0, 0.0, 0.0)),
        })
    return traces

if 'curr_frame_idx' not in globals():
    curr_frame_idx = FRAME_PAIR_INDEX + FRAME_STEP
if 'curr_frame' not in globals():
    curr_frame = read_frame(VIDEO_PATH, curr_frame_idx)
if 'focus_box' not in globals():
    focus_box = None
if 'representative_record' not in globals():
    representative_record = None
if 'optimized_records' not in globals():
    optimized_records = []

motion_window = list(range(max(0, curr_frame_idx - 2), min(curr_frame_idx + 3, globals().get('CAP_FRAME_COUNT', curr_frame_idx + 3))))
records_by_frame = {int(item.get('frame', -1)): item for item in optimized_records if item.get('frame') is not None}
if motion_window and len(motion_window) >= 2:
    fallback_roi = focus_box if focus_box is not None else (0, 0, curr_frame.shape[1], curr_frame.shape[0])
    selected_motion_roi = choose_motion_roi(
        (curr_frame.shape[0], curr_frame.shape[1]),
        record=(representative_record if representative_record is not None else None),
        fallback_roi=fallback_roi,
    )
    motion_history = summarize_motion_history(motion_window, roi=selected_motion_roi)
    motion_df = pd.DataFrame(motion_history)
    if not motion_df.empty:
        print(motion_df[['frame_a', 'frame_b', 'translation_px', 'translation_m', 'rotation_proxy_deg', 'mean_speed_px']].to_string(index=False))
        fig, axes = plt.subplots(2, 2, figsize=(16, 10))
        axes[0, 0].plot(motion_df['frame_b'], motion_df['translation_m'], marker='o', linewidth=2, color='tab:blue')
        axes[0, 0].set_title('3D motion proxy over time')
        axes[0, 0].set_xlabel('Frame index')
        axes[0, 0].set_ylabel('Translation proxy (m)')
        axes[0, 1].plot(motion_df['frame_b'], motion_df['rotation_proxy_deg'], marker='s', linewidth=2, color='tab:orange')
        axes[0, 1].set_title('Rotation proxy over time')
        axes[0, 1].set_xlabel('Frame index')
        axes[0, 1].set_ylabel('Rotation (deg)')
        object_series = summarize_object_stats(motion_window, records_by_frame)
        if object_series:
            object_df = pd.DataFrame(object_series)
            axes[1, 0].plot(object_df['frame'], object_df['vehicle_count'], marker='o', linewidth=2, color='tab:green')
            axes[1, 0].set_title('Detected vehicle count over time')
            axes[1, 0].set_xlabel('Frame index')
            axes[1, 0].set_ylabel('Vehicle count')
            hist_bins = max(1, min(8, int(object_df['vehicle_count'].max() + 1)))
            axes[1, 1].hist(object_df['vehicle_count'], bins=hist_bins, color='tab:purple', alpha=0.8)
            axes[1, 1].set_title('Histogram of detected vehicle counts')
            axes[1, 1].set_xlabel('Vehicles per frame')
            axes[1, 1].set_ylabel('Frames')
        else:
            axes[1, 0].text(0.5, 0.5, 'No detection records for this window', ha='center', va='center', transform=axes[1, 0].transAxes)
            axes[1, 1].text(0.5, 0.5, 'No detection histogram available', ha='center', va='center', transform=axes[1, 1].transAxes)
        for ax in axes.flat:
            ax.grid(True, alpha=0.25)
        plt.tight_layout()
        plt.show()
        print(f'Motion ROI used: {selected_motion_roi}')
    else:
        print('No usable motion samples for a kinetic trace in this frame window.')
else:
    print('Not enough frames for a meaningful per-step kinetic trace.')

print('ROI emphasis: vehicle-centered ROI with static-landmark fallback; motion summary includes a 3D proxy and object-count time series.')


In [ ]:
def load_jsonl_records(path: Path) -> list[dict]:
    records = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

def overlay_record_on_frame(frame: np.ndarray, record: dict) -> np.ndarray:
    canvas = frame.copy()
    tint = canvas.copy()
    for vehicle in record.get('vehicle_boxes', []):
        x1, y1, x2, y2 = [int(v) for v in vehicle.get('xyxy', [0, 0, 0, 0])]
        cv2.rectangle(tint, (x1, y1), (x2, y2), (0, 180, 180), -1)
        cv2.rectangle(canvas, (x1, y1), (x2, y2), (0, 255, 255), 2)
    for item in record.get('ocr', []):
        x1, y1, x2, y2 = [int(v) for v in item.get('xyxy', [0, 0, 0, 0])]
        cv2.rectangle(tint, (x1, y1), (x2, y2), (50, 170, 50), -1)
        label = f"{item.get('ocr_text', '') or '<empty>'} | conf={float(item.get('ocr_conf', 0.0)):.2f}"
        cv2.rectangle(canvas, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(canvas, label, (x1, max(18, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1, cv2.LINE_AA)
    cv2.addWeighted(tint, 0.18, canvas, 0.82, 0, canvas)
    return canvas

def crop_gallery(frame: np.ndarray, boxes: list[list[int]], title: str, max_items: int = 5):
    subset = boxes[:max_items]
    if not subset:
        print('No boxes available for gallery')
        return
    fig, axes = plt.subplots(1, len(subset), figsize=(4 * len(subset), 4))
    if len(subset) == 1:
        axes = [axes]
    for ax, xyxy in zip(axes, subset):
        x1, y1, x2, y2 = [int(v) for v in xyxy]
        crop = frame[max(0, y1):max(0, y2), max(0, x1):max(0, x2)]
        ax.imshow(as_rgb(crop))
        ax.set_title(f'{x2 - x1}x{y2 - y1}')
        ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

def summarize_vehicle_boxes(record: dict) -> dict:
    vehicles = record.get('vehicle_boxes', [])
    confidences = [float(item.get('conf', 0.0)) for item in vehicles]
    return {
        'vehicle_count': len(vehicles),
        'mean_conf': float(np.mean(confidences)) if confidences else 0.0,
        'frame_diff_score': float(record.get('frame_diff_score', 0.35)) if record.get('frame_diff_score', None) is not None else 0.35,
    }

def select_representative_record(records: list[dict], min_vehicle_count: int = 2, max_vehicle_count: int = 10) -> dict | None:
    candidates = []
    for record in records:
        stats = summarize_vehicle_boxes(record)
        vehicle_count = stats['vehicle_count']
        if vehicle_count < min_vehicle_count or vehicle_count > max_vehicle_count:
            continue
        vehicle_balance = 1.0 - min(1.0, abs(vehicle_count - 4) / 6.0)
        score = 2.5 * vehicle_balance + 2.0 * stats['mean_conf'] - 1.5 * stats['frame_diff_score']
        candidates.append((score, record))
    if not candidates:
        return None
    candidates.sort(key=lambda item: item[0], reverse=True)
    return candidates[0][1]

def nearest_record_by_frame(records: list[dict], frame_idx: int) -> dict | None:
    if not records:
        return None
    return min(records, key=lambda item: abs(int(item.get('frame', 0)) - int(frame_idx)))

def build_feature_extractor():
    if hasattr(cv2, 'SIFT_create'):
        return cv2.SIFT_create(), cv2.NORM_L2, 'SIFT'
    return cv2.ORB_create(nfeatures=1000), cv2.NORM_HAMMING, 'ORB'

def match_vehicle_features(prev_frame: np.ndarray, curr_frame: np.ndarray, prev_xyxy: list[int], curr_xyxy: list[int], max_matches: int = 40) -> dict:
    px1, py1, px2, py2 = [int(v) for v in prev_xyxy]
    cx1, cy1, cx2, cy2 = [int(v) for v in curr_xyxy]
    prev_crop = prev_frame[max(0, py1):max(0, py2), max(0, px1):max(0, px2)]
    curr_crop = curr_frame[max(0, cy1):max(0, cy2), max(0, cx1):max(0, cx2)]
    extractor, norm_type, method = build_feature_extractor()
    kp_prev, desc_prev = extractor.detectAndCompute(prev_crop, None)
    kp_curr, desc_curr = extractor.detectAndCompute(curr_crop, None)
    if desc_prev is None or desc_curr is None or not kp_prev or not kp_curr:
        return {
            'method': method,
            'keypoints_prev': len(kp_prev or []),
            'keypoints_curr': len(kp_curr or []),
            'match_count': 0,
            'match_image': np.concatenate([prev_crop, curr_crop], axis=1) if prev_crop.size and curr_crop.size else np.zeros((10, 10, 3), dtype=np.uint8),
            'prev_crop': prev_crop,
            'curr_crop': curr_crop,
        }
    if norm_type == cv2.NORM_L2:
        matcher = cv2.BFMatcher(norm_type)
        raw_matches = matcher.knnMatch(desc_prev, desc_curr, k=2)
        good_matches = [m for m, n in raw_matches if n is not None and m.distance < 0.75 * n.distance]
    else:
        matcher = cv2.BFMatcher(norm_type, crossCheck=True)
        good_matches = sorted(matcher.match(desc_prev, desc_curr), key=lambda m: m.distance)
    good_matches = good_matches[:max_matches]
    match_image = cv2.drawMatches(prev_crop, kp_prev, curr_crop, kp_curr, good_matches, None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    return {
        'method': method,
        'keypoints_prev': len(kp_prev),
        'keypoints_curr': len(kp_curr),
        'match_count': len(good_matches),
        'match_image': match_image,
        'prev_crop': prev_crop,
        'curr_crop': curr_crop,
    }

In [ ]:
def choose_motion_roi(frame_shape: tuple[int, int], record: dict | None = None, fallback_roi: tuple[int, int, int, int] | None = None, padding: int = 30) -> tuple[int, int, int, int]:
    """Prefer a vehicle-centered ROI, then fall back to a static landmark or full frame."""
    if record is not None:
        vehicles = record.get('vehicle_boxes', []) or []
        if vehicles:
            best = max(vehicles, key=lambda item: float(item.get('conf', 0.0)) * max(1.0, float(item.get('area', 1.0))))
            xyxy = best.get('xyxy', [0, 0, 0, 0])
            x1, y1, x2, y2 = [int(v) for v in xyxy]
            x1 = max(0, x1 - padding)
            y1 = max(0, y1 - padding)
            x2 = min(frame_shape[1], x2 + padding)
            y2 = min(frame_shape[0], y2 + padding)
            return (x1, y1, x2, y2)
    if fallback_roi is not None:
        return fallback_roi
    return (0, 0, frame_shape[1], frame_shape[0])


def summarize_object_stats(frame_indices: list[int], records_by_frame: dict[int, dict]) -> list[dict]:
    rows = []
    for idx in frame_indices:
        record = records_by_frame.get(int(idx))
        if record is None:
            rows.append({
                'frame': idx,
                'vehicle_count': 0,
                'mean_conf': 0.0,
                'mean_width': 0.0,
                'mean_height': 0.0,
            })
            continue
        vehicles = record.get('vehicle_boxes', []) or []
        widths = []
        heights = []
        confs = []
        for vehicle in vehicles:
            x1, y1, x2, y2 = [int(v) for v in vehicle.get('xyxy', [0, 0, 0, 0])]
            widths.append(max(0, x2 - x1))
            heights.append(max(0, y2 - y1))
            confs.append(float(vehicle.get('conf', 0.0)))
        rows.append({
            'frame': idx,
            'vehicle_count': len(vehicles),
            'mean_conf': float(np.mean(confs)) if confs else 0.0,
            'mean_width': float(np.mean(widths)) if widths else 0.0,
            'mean_height': float(np.mean(heights)) if heights else 0.0,
        })
    return rows


def summarize_motion_history(frame_indices: list[int], roi: tuple[int, int, int, int] | None = None, motion_scale: float = 35.0) -> list[dict]:
    """Compute per-step translation, rotation, and 3D-style motion proxies across a short frame sequence."""
    traces = []
    for idx in range(1, len(frame_indices)):
        prev_idx = frame_indices[idx - 1]
        curr_idx = frame_indices[idx]
        prev_frame = read_frame(VIDEO_PATH, prev_idx)
        curr_frame = read_frame(VIDEO_PATH, curr_idx)
        if prev_frame is None or curr_frame is None:
            continue
        metrics = rov.estimate_optical_flow_kinematics(
            prev_frame,
            curr_frame,
            roi=roi,
            motion_scale=motion_scale,
            vector_step=10,
            altitude_m=35.0,
        )
        traces.append({
            'frame_a': prev_idx,
            'frame_b': curr_idx,
            'translation_px': metrics['translation_px'],
            'translation_m': metrics.get('translation_m', 0.0),
            'rotation_proxy_deg': metrics['rotation_proxy_deg'],
            'mean_speed_px': metrics['mean_speed_px'],
            'translation_vector_px': metrics['translation_vector_px'],
            'translation_vector_m': metrics.get('translation_vector_m', (0.0, 0.0, 0.0)),
        })
    return traces

if 'curr_frame_idx' not in globals():
    curr_frame_idx = FRAME_PAIR_INDEX + FRAME_STEP
if 'curr_frame' not in globals():
    curr_frame = read_frame(VIDEO_PATH, curr_frame_idx)
if 'focus_box' not in globals():
    focus_box = None
if 'representative_record' not in globals():
    representative_record = None
if 'optimized_records' not in globals():
    optimized_records = []

motion_window = list(range(max(0, curr_frame_idx - 2), min(curr_frame_idx + 3, globals().get('CAP_FRAME_COUNT', curr_frame_idx + 3))))
records_by_frame = {int(item.get('frame', -1)): item for item in optimized_records if item.get('frame') is not None}
if motion_window and len(motion_window) >= 2:
    fallback_roi = focus_box if focus_box is not None else (0, 0, curr_frame.shape[1], curr_frame.shape[0])
    selected_motion_roi = choose_motion_roi(
        (curr_frame.shape[0], curr_frame.shape[1]),
        record=(representative_record if representative_record is not None else None),
        fallback_roi=fallback_roi,
    )
    motion_history = summarize_motion_history(motion_window, roi=selected_motion_roi)
    motion_df = pd.DataFrame(motion_history)
    if not motion_df.empty:
        print(motion_df[['frame_a', 'frame_b', 'translation_px', 'translation_m', 'rotation_proxy_deg', 'mean_speed_px']].to_string(index=False))
        fig, axes = plt.subplots(2, 2, figsize=(16, 10))
        axes[0, 0].plot(motion_df['frame_b'], motion_df['translation_m'], marker='o', linewidth=2, color='tab:blue')
        axes[0, 0].set_title('3D motion proxy over time')
        axes[0, 0].set_xlabel('Frame index')
        axes[0, 0].set_ylabel('Translation proxy (m)')
        axes[0, 1].plot(motion_df['frame_b'], motion_df['rotation_proxy_deg'], marker='s', linewidth=2, color='tab:orange')
        axes[0, 1].set_title('Rotation proxy over time')
        axes[0, 1].set_xlabel('Frame index')
        axes[0, 1].set_ylabel('Rotation (deg)')
        object_series = summarize_object_stats(motion_window, records_by_frame)
        if object_series:
            object_df = pd.DataFrame(object_series)
            axes[1, 0].plot(object_df['frame'], object_df['vehicle_count'], marker='o', linewidth=2, color='tab:green')
            axes[1, 0].set_title('Detected vehicle count over time')
            axes[1, 0].set_xlabel('Frame index')
            axes[1, 0].set_ylabel('Vehicle count')
            hist_bins = max(1, min(8, int(object_df['vehicle_count'].max() + 1)))
            axes[1, 1].hist(object_df['vehicle_count'], bins=hist_bins, color='tab:purple', alpha=0.8)
            axes[1, 1].set_title('Histogram of detected vehicle counts')
            axes[1, 1].set_xlabel('Vehicles per frame')
            axes[1, 1].set_ylabel('Frames')
        else:
            axes[1, 0].text(0.5, 0.5, 'No detection records for this window', ha='center', va='center', transform=axes[1, 0].transAxes)
            axes[1, 1].text(0.5, 0.5, 'No detection histogram available', ha='center', va='center', transform=axes[1, 1].transAxes)
        for ax in axes.flat:
            ax.grid(True, alpha=0.25)
        plt.tight_layout()
        plt.show()
        print(f'Motion ROI used: {selected_motion_roi}')
    else:
        print('No usable motion samples for a kinetic trace in this frame window.')
else:
    print('Not enough frames for a meaningful per-step kinetic trace.')

print('ROI emphasis: vehicle-centered ROI with static-landmark fallback; motion summary includes a 3D proxy and object-count time series.')


In [ ]:
if 'curr_frame_idx' not in globals():
    curr_frame_idx = FRAME_PAIR_INDEX + FRAME_STEP
if 'curr_frame' not in globals():
    curr_frame = read_frame(VIDEO_PATH, curr_frame_idx)
if 'prev_frame_idx' not in globals():
    prev_frame_idx = max(0, curr_frame_idx - FRAME_STEP)
if 'prev_frame' not in globals():
    prev_frame = read_frame(VIDEO_PATH, prev_frame_idx)
if 'next_frame_idx' not in globals():
    next_frame_idx = min(curr_frame_idx + FRAME_STEP, globals().get('CAP_FRAME_COUNT', curr_frame_idx + FRAME_STEP))
if 'next_frame' not in globals():
    next_frame = read_frame(VIDEO_PATH, next_frame_idx)
if 'focus_box' not in globals():
    focus_box = None
if 'selected_motion_roi' not in globals():
    selected_motion_roi = choose_motion_roi(
        (curr_frame.shape[0], curr_frame.shape[1]),
        record=(representative_record if 'representative_record' in globals() and representative_record is not None else None),
        fallback_roi=(0, 0, curr_frame.shape[1], curr_frame.shape[0]),
    )
if selected_motion_roi is not None and focus_box is None:
    focus_box = selected_motion_roi
if 'motion_df' not in globals():
    motion_window = list(range(max(0, curr_frame_idx - 2), min(curr_frame_idx + 3, globals().get('CAP_FRAME_COUNT', curr_frame_idx + 3))))
    motion_history = summarize_motion_history(motion_window, roi=selected_motion_roi)
    motion_df = pd.DataFrame(motion_history)

if motion_df.empty:
    print('No motion samples are available for the selected ROI.')
else:
    flow_prev_curr, flow_prev_curr_summary = render_optical_flow(prev_frame, curr_frame, box=selected_motion_roi, motion_scale=35.0)
    flow_curr_next, flow_curr_next_summary = render_optical_flow(curr_frame, next_frame, box=selected_motion_roi, motion_scale=35.0)
    flow_focus, flow_focus_summary = render_optical_flow(prev_frame, curr_frame, box=selected_motion_roi, motion_scale=35.0)
    print({
        'prev_to_curr_mean_motion_px': round(float(flow_prev_curr_summary['mean_motion_px']), 4),
        'curr_to_next_mean_motion_px': round(float(flow_curr_next_summary['mean_motion_px']), 4),
        'focus_roi_mean_motion_px': round(float(flow_focus_summary['mean_motion_px']), 4),
        'trajectory_mean_mag_px': round(float(motion_df['translation_m'].mean()), 4),
    })

    fig, axes = plt.subplots(2, 3, figsize=(21, 11))
    axes[0, 0].imshow(as_rgb(prev_frame))
    axes[0, 0].set_title(f'Previous frame {prev_frame_idx}')
    axes[0, 0].axis('off')
    axes[0, 1].imshow(as_rgb(curr_frame))
    axes[0, 1].set_title(f'Current frame {curr_frame_idx}')
    axes[0, 1].axis('off')
    axes[0, 2].imshow(as_rgb(next_frame))
    axes[0, 2].set_title(f'Next frame {next_frame_idx}')
    axes[0, 2].axis('off')
    axes[1, 0].imshow(as_rgb(flow_prev_curr))
    axes[1, 0].set_title(f"Flow prev→curr | mean={flow_prev_curr_summary['mean_motion_px']:.2f}px")
    axes[1, 0].axis('off')
    axes[1, 1].imshow(as_rgb(flow_curr_next))
    axes[1, 1].set_title(f"Flow curr→next | mean={flow_curr_next_summary['mean_motion_px']:.2f}px")
    axes[1, 1].axis('off')
    axes[1, 2].imshow(as_rgb(flow_focus))
    axes[1, 2].set_title(f"ROI flow context | mean={flow_focus_summary['mean_motion_px']:.2f}px")
    axes[1, 2].axis('off')
    plt.tight_layout()
    plt.show()

    if selected_motion_roi is not None:
        print('Focused motion ROI:', selected_motion_roi)


## Real Overlay Failure Case

This section inspects the existing mission JSONL directly. It answers the practical question: were the green boxes actually tight plate regions, or were they oversized candidate boxes that merely survived the overlay filter?

For the older output pass, many records have:

- zero `vehicle_boxes`
- several large `plate_boxes` and `ocr` boxes
- empty `ocr_text`

That is the exact pattern that makes the overlay look misleading.

In [ ]:
## Optical Flow Motion Check

## This motion view complements the frame-difference heatmap by estimating local motion vectors between consecutive frames. The goal is to check whether the scene is moving in a coherent pattern, or whether the vehicle ROI contains mostly static content that should be filtered out before OCR reuse decisions.


In [ ]:
legacy_jsonl = OUTPUT_DIR / f"{VIDEO_PATH.stem}.detections.jsonl"
optimized_jsonl = OUTPUT_DIR / 'detections' / f"{VIDEO_PATH.stem}.detections.jsonl"

legacy_records = load_jsonl_records(legacy_jsonl) if legacy_jsonl.exists() else []
optimized_records = load_jsonl_records(optimized_jsonl) if optimized_jsonl.exists() else []

print('legacy_jsonl_exists', legacy_jsonl.exists(), 'records', len(legacy_records))
print('optimized_jsonl_exists', optimized_jsonl.exists(), 'records', len(optimized_records))

def suspicious_record(records: list[dict]) -> dict | None:
    for record in records:
        vehicles = record.get('vehicle_boxes', [])
        ocr_items = record.get('ocr', [])
        if not vehicles and len(ocr_items) >= 3:
            return record
    return records[0] if records else None

legacy_bad = suspicious_record(legacy_records)
legacy_bad

## Automatic Sample Selection

The notebook now prefers a more informative frame pair from the optimized detections output. The goal is to avoid weak examples and bias the walkthrough toward frames that contain real vehicles, usable re-identification context, and clearer box semantics.

In [ ]:
representative_record = select_representative_record(optimized_records, min_vehicle_count=2)
if representative_record is not None:
    curr_frame_idx = int(representative_record['frame'])
    prev_frame_idx = max(0, curr_frame_idx - FRAME_STEP)
    prev_frame = read_frame(VIDEO_PATH, prev_frame_idx)
    curr_frame = read_frame(VIDEO_PATH, curr_frame_idx)
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
    frame_diff = rcp.frame_difference_score(prev_gray, curr_gray)
    diff_vis = cv2.absdiff(prev_gray, curr_gray)
    diff_vis = cv2.applyColorMap(cv2.normalize(diff_vis, None, 0, 255, cv2.NORM_MINMAX), cv2.COLORMAP_INFERNO)
    print({
        'selected_frame': curr_frame_idx,
        'previous_frame': prev_frame_idx,
        'vehicle_summary': summarize_vehicle_boxes(representative_record),
    })
    show_side_by_side(curr_frame, diff_vis, f'Auto-selected frame {curr_frame_idx}', 'Difference heatmap for selected pair')
else:
    print('No optimized record with enough vehicles was found; keeping the manual frame selection.')

In [ ]:
if legacy_bad is None:
    print('No legacy detection record was found.')
else:
    legacy_frame = read_frame(VIDEO_PATH, int(legacy_bad['frame']))
    legacy_overlay = overlay_record_on_frame(legacy_frame, legacy_bad)
    print({
        'frame': legacy_bad['frame'],
        'vehicle_count': len(legacy_bad.get('vehicle_boxes', [])),
        'ocr_count': len(legacy_bad.get('ocr', [])),
        'plate_box_count': len(legacy_bad.get('plate_boxes', [])),
        'all_empty_ocr_text': all(not str(item.get('ocr_text') or '').strip() for item in legacy_bad.get('ocr', [])),
    })
    show_side_by_side(legacy_frame, legacy_overlay, 'Raw source frame', 'Legacy overlay semantics on that frame')
    crop_gallery(
        legacy_frame,
        [item['xyxy'] for item in legacy_bad.get('ocr', []) if 'xyxy' in item],
        'Crops behind green OCR boxes from the old output',
    )

In [ ]:
models = rcp.LazyModels(DEVICE if cv2.cuda.getCudaEnabledDeviceCount() > 0 else 'cpu')
vehicle_result = models.vehicle_model().predict([curr_frame], device=DEVICE, verbose=False)[0]
vehicle_boxes = []
if getattr(vehicle_result, 'boxes', None) is not None:
    for box in vehicle_result.boxes:
        cls_id = int(box.cls.item())
        if cls_id not in {2, 3, 5, 7}:
            continue
        xyxy = [int(x) for x in box.xyxy[0].tolist()]
        vehicle_boxes.append({
            'xyxy': xyxy,
            'class_name': lps.vehicle_class_name_from_id(cls_id),
            'conf': round(float(box.conf.item()), 4),
        })

vehicle_boxes = sorted(vehicle_boxes, key=lambda item: item['conf'], reverse=True)[:8]
vehicle_overlay = draw_boxes(curr_frame, vehicle_boxes, (0, 255, 0), label_key='class_name')
print('vehicle_count', len(vehicle_boxes))
show_side_by_side(curr_frame, vehicle_overlay, 'Original frame', 'Vehicle detections')
pd.DataFrame(vehicle_boxes)

In [ ]:
if not vehicle_boxes:
    raise RuntimeError('No vehicles were detected on the selected frame. Try a different FRAME_PAIR_INDEX.')

focus_vehicle = vehicle_boxes[0]
candidate_boxes = lps.extract_plate_candidates_from_vehicle(curr_frame, focus_vehicle['xyxy'], frame_shape=curr_frame.shape, max_candidates=3)
candidate_overlay = draw_boxes(curr_frame, [{'xyxy': focus_vehicle['xyxy']}], (0, 255, 0))
candidate_overlay = draw_boxes(candidate_overlay, candidate_boxes, (255, 255, 0))

x1, y1, x2, y2 = focus_vehicle['xyxy']
vehicle_crop = curr_frame[y1:y2, x1:x2]
print('focus_vehicle', focus_vehicle)
print('candidate_boxes', candidate_boxes)
show_side_by_side(candidate_overlay, vehicle_crop, 'Vehicle + lower-band plate proposals', 'Focused vehicle crop')

## Plate Segmentation And Rectification

The notebook now shows the separate plate-localization method inside the vehicle ROI. Instead of trusting only lower-band proposals, it first looks for plate-like quadrilaterals, then rectifies them into a straighter crop for OCR.

This is the main path intended to reduce failures on angled rear plates.

In [ ]:
segmented_candidates = [item for item in candidate_boxes if item.get('source') == 'vehicle_plate_segment']
heuristic_candidates = [item for item in candidate_boxes if item.get('source') != 'vehicle_plate_segment']

def draw_candidate_quads(frame: np.ndarray, candidates: list[dict], color: tuple[int, int, int]) -> np.ndarray:
    canvas = frame.copy()
    for item in candidates:
        quad = item.get('quad')
        if quad:
            pts = np.array(quad, dtype=np.int32).reshape((-1, 1, 2))
            cv2.polylines(canvas, [pts], True, color, 2)
        else:
            x1, y1, x2, y2 = [int(v) for v in item['xyxy']]
            cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 2)
    return canvas

segmentation_overlay = draw_boxes(curr_frame, [{'xyxy': focus_vehicle['xyxy']}], (0, 255, 255))
segmentation_overlay = draw_candidate_quads(segmentation_overlay, segmented_candidates, (255, 120, 0))
segmentation_overlay = draw_candidate_quads(segmentation_overlay, heuristic_candidates, (255, 255, 0))

print({
    'segmented_candidate_count': len(segmented_candidates),
    'heuristic_candidate_count': len(heuristic_candidates),
    'candidate_sources': [item.get('source', 'unknown') for item in candidate_boxes],
})

show_side_by_side(
    segmentation_overlay,
    vehicle_crop,
    'Orange = segmented plate quads, Yellow = heuristic fallback, Cyan = vehicle box',
    'Focused vehicle crop',
)

if segmented_candidates:
    rectified_crops = [lps.extract_plate_crop(curr_frame, item) for item in segmented_candidates]
    fig, axes = plt.subplots(1, len(rectified_crops), figsize=(5 * len(rectified_crops), 4))
    if len(rectified_crops) == 1:
        axes = [axes]
    for ax, crop, item in zip(axes, rectified_crops, segmented_candidates):
        ax.imshow(as_rgb(crop))
        ax.set_title(f"{item.get('source')} | {crop.shape[1]}x{crop.shape[0]}")
        ax.axis('off')
    fig.suptitle('Rectified plate crops produced by the segmentation path')
    plt.tight_layout()
    plt.show()
else:
    print('No segmented quadrilateral candidate was found on this frame; heuristic fallback remains active.')

In [ ]:
legacy_comparison_record = nearest_record_by_frame(legacy_records, curr_frame_idx)
if legacy_comparison_record is None:
    print('No legacy record is available for comparison.')
else:
    comparison_overlay = draw_boxes(
        curr_frame,
        [{'xyxy': box['xyxy']} for box in legacy_comparison_record.get('ocr', []) if 'xyxy' in box],
        (0, 255, 0),
    )
    comparison_overlay = draw_boxes(comparison_overlay, candidate_boxes, (255, 255, 0))
    print({
        'current_frame': curr_frame_idx,
        'legacy_frame_used': int(legacy_comparison_record.get('frame', -1)),
        'legacy_ocr_count': len(legacy_comparison_record.get('ocr', [])),
        'candidate_count': len(candidate_boxes),
    })
    show_side_by_side(
        comparison_overlay,
        candidate_overlay,
        'Green = nearest legacy OCR boxes, Yellow = current candidate boxes',
        'Vehicle + current lower-band plate proposals',
    )

In [ ]:
tuning = rcp.RuntimeTuning(
    use_full_frame_plate_detector=False,
    ocr_frame_interval=3,
    max_plate_boxes=6,
    max_ocr_crops=3,
    min_plate_crop_area=400,
)
ocr_items, ocr_attempts = rcp.run_plate_ocr(curr_frame, candidate_boxes, models, tuning)
ocr_df = pd.DataFrame(ocr_items) if ocr_items else pd.DataFrame(columns=['xyxy', 'ocr_text', 'ocr_conf', 'plate_conf', 'sharpness'])
print({'ocr_attempts': ocr_attempts, 'ocr_result_count': len(ocr_items)})
ocr_df[['xyxy', 'ocr_text', 'ocr_conf', 'plate_conf', 'sharpness']] if not ocr_df.empty else ocr_df

In [ ]:
prev_result = models.vehicle_model().predict([prev_frame], device=DEVICE, verbose=False)[0]
prev_vehicle_boxes = []
if getattr(prev_result, 'boxes', None) is not None:
    for box in prev_result.boxes:
        cls_id = int(box.cls.item())
        if cls_id not in {2, 3, 5, 7}:
            continue
        prev_vehicle_boxes.append({
            'xyxy': [int(x) for x in box.xyxy[0].tolist()],
            'class_name': lps.vehicle_class_name_from_id(cls_id),
            'conf': round(float(box.conf.item()), 4),
        })

previous_tracks, next_track_id = rcp.assign_vehicle_tracks(prev_vehicle_boxes[:8], prev_frame, [], tuning, 1)
current_tracks, _ = rcp.assign_vehicle_tracks(vehicle_boxes, curr_frame, previous_tracks, tuning, next_track_id)
track_pairs = []
for current_vehicle in vehicle_boxes:
    track_id = int(current_vehicle.get('track_id', 0))
    previous_track = next((track for track in previous_tracks if track.track_id == track_id), None)
    if previous_track is None:
        continue
    similarity = rcp.signature_similarity(rcp.vehicle_signature(curr_frame, current_vehicle['xyxy']), previous_track.signature)
    overlap = rcp.bbox_iou(current_vehicle['xyxy'], previous_track.xyxy)
    track_pairs.append({
        'track_id': track_id,
        'class_name': current_vehicle['class_name'],
        'iou': round(overlap, 4),
        'signature_similarity': round(similarity, 4),
        'frame_diff_score': round(frame_diff, 6),
        'ocr_can_be_reused': bool(frame_diff <= tuning.reid_frame_diff_threshold and overlap >= tuning.reid_match_iou_threshold and similarity >= tuning.reid_similarity_threshold),
    })

pd.DataFrame(track_pairs).sort_values(['ocr_can_be_reused', 'signature_similarity'], ascending=[False, False])

## Feature Matching For Re-Identification

This section visualizes re-identification directly. It compares the matched vehicle crop from the previous frame to the current frame, then draws feature correspondences using SIFT when available and ORB otherwise.

The point is not just to compute a similarity score, but to show whether the object is visually stable enough that a heavy OCR or plate-detection pass can be skipped and the prior track state reused.

In [ ]:
track_df = pd.DataFrame(track_pairs).sort_values(['ocr_can_be_reused', 'signature_similarity', 'iou'], ascending=[False, False, False])
display(track_df)

if track_df.empty:
    print('No matched vehicle tracks were found for feature-matching visualization.')
else:
    best_track = track_df.iloc[0].to_dict()
    best_track_id = int(best_track['track_id'])
    previous_track = next((track for track in previous_tracks if track.track_id == best_track_id), None)
    current_vehicle = next((vehicle for vehicle in vehicle_boxes if int(vehicle.get('track_id', -1)) == best_track_id), None)
    if previous_track is None or current_vehicle is None:
        print('Unable to resolve the previous/current vehicle pair for the best track.')
    else:
        match_result = match_vehicle_features(prev_frame, curr_frame, previous_track.xyxy, current_vehicle['xyxy'])
        print({
            'track_id': best_track_id,
            'feature_method': match_result['method'],
            'keypoints_prev': match_result['keypoints_prev'],
            'keypoints_curr': match_result['keypoints_curr'],
            'feature_matches': match_result['match_count'],
            'iou': float(best_track['iou']),
            'signature_similarity': float(best_track['signature_similarity']),
            'frame_diff_score': float(best_track['frame_diff_score']),
            'ocr_can_be_reused': bool(best_track['ocr_can_be_reused']),
        })
        show_side_by_side(
            match_result['prev_crop'],
            match_result['curr_crop'],
            'Previous-frame matched vehicle crop',
            'Current-frame matched vehicle crop',
        )
        plt.figure(figsize=(18, 7))
        plt.imshow(as_rgb(match_result['match_image']))
        plt.title('Feature matches for re-identification decision support')
        plt.axis('off')
        plt.tight_layout()
        plt.show()

In [ ]:
fused_dir = OUTPUT_DIR / 'fused'
summary_path = OUTPUT_DIR / 'lp_vehicle_report.summary.json'
geojson_path = OUTPUT_DIR / 'lp_vehicle_report.geojson'

if fused_dir.exists() and list(fused_dir.glob('*.fused.csv')):
    fused_df = bdr.load_fused_observations(fused_dir)
    fused_df['review_status'] = fused_df.apply(bdr.review_status, axis=1)
    display_cols = [
        'video', 'timestamp', 'plate_text', 'fused_confidence', 'vehicle_type',
        'vehicle_color', 'support_frames', 'geolocation_mode', 'review_status',
        'latitude', 'longitude'
    ]
    display(fused_df[display_cols].sort_values('fused_confidence', ascending=False).head(15))
    print('summary_path_exists', summary_path.exists())
    print('geojson_path_exists', geojson_path.exists())
    if summary_path.exists():
        print(json.loads(summary_path.read_text(encoding='utf-8')))
else:
    print('No fused outputs are available yet. Run the full 5-stage pipeline first.')

## Module 4 · Integration: sync + fused geospatial report

`sync_detections_with_srt.py` stamps every detection with the nearest SRT
record; `fuse_plate_observations.py` merges repeated observations of the same
vehicle into one geospatial record with `geolocation_mode` and
`geo_spread_m`. The cell below renders the integrated map and fusion
statistics, and exports them for the apps.



In [ ]:
fused_dirs = sorted(ROOT.glob('output/cv/*/fused/*.fused.csv'))
geotagged_csvs = sorted(ROOT.glob('output/cv/*/synced/*.geotagged.csv'))
report_geojsons = sorted(ROOT.glob('output/cv/*/lp_vehicle_report.geojson'))

if fused_dirs and geotagged_csvs:
    fused_path = fused_dirs[0]
    geo_path = geotagged_csvs[0]
    fused_df = pd.read_csv(fused_path)
    geo_df = pd.read_csv(geo_path)
    print('Fused records:', len(fused_df), '| Geotagged detections:', len(geo_df))
    print('Fused columns:', list(fused_df.columns)[:14])

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    axes[0].scatter(geo_df['longitude'], geo_df['latitude'], s=3, c='#94a3b8', label='drone pose (SRT sync)')
    has_ll = 'latitude' in fused_df.columns and 'longitude' in fused_df.columns
    if has_ll and not fused_df[['latitude', 'longitude']].dropna().empty:
        axes[0].scatter(fused_df['longitude'], fused_df['latitude'], s=45, c='tab:red', marker='^', label='fused object points')
    axes[0].set_title('Integrated map: drone path vs fused object points')
    axes[0].set_xlabel('longitude'); axes[0].set_ylabel('latitude')
    axes[0].legend(); axes[0].grid(True, alpha=0.25)

    if 'fused_confidence' in fused_df.columns:
        fused_df['fused_confidence'].hist(bins=20, ax=axes[1])
        axes[1].set_title('Fused plate confidence distribution')
        axes[1].set_xlabel('confidence')
    elif 'plate_confidence' in fused_df.columns:
        fused_df['plate_confidence'].hist(bins=20, ax=axes[1])
        axes[1].set_title('Plate confidence distribution')
        axes[1].set_xlabel('confidence')
    else:
        axes[1].text(0.5, 0.5, 'no confidence column', ha='center', va='center', transform=axes[1].transAxes)
    axes[1].grid(True, alpha=0.25)
    fig.suptitle('Module 4: integrated sync + fusion report')
    fig.tight_layout()
    fig.savefig(LAB_DIR / 'module4_integration_fusion.png', dpi=130)
    plt.show()
else:
    print('No fused/geotagged outputs yet. Run scripts/run_lp_geospatial_pipeline.sh on a mission first.')

if report_geojsons:
    report_path = report_geojsons[0]
    with report_path.open('r', encoding='utf-8') as f:
        gj = json.load(f)
    features = gj.get('features', [])
    modes = {}
    for feat in features:
        props = feat.get('properties', {})
        modes[props.get('geolocation_mode', 'unknown')] = modes.get(props.get('geolocation_mode', 'unknown'), 0) + 1
    print('Report GeoJSON:', report_path.name, '| features:', len(features), '| geolocation modes:', modes)



## Module 5 · render_overlay_video (motion)

The overlay renderer composites the detection HUD with telemetry. The motion
cell below reuses its optical-flow engine to produce the workshop motion view
and exports the frame for the apps. This is the visualization that will feed
the "Live preview performance demo" in the desktop app.



In [ ]:
from pathlib import Path as _Path

_demo_idx = FRAME_PAIR_INDEX if 'FRAME_PAIR_INDEX' in globals() else 240
_prev_idx = max(0, _demo_idx - FRAME_STEP)
_prev_frame = read_frame(VIDEO_PATH, _prev_idx)
_curr_frame = read_frame(VIDEO_PATH, _demo_idx)
_roi = choose_motion_roi((_curr_frame.shape[0], _curr_frame.shape[1]), padding=30)
_flow_overlay, _flow_summary = render_optical_flow(_prev_frame, _curr_frame, box=_roi, motion_scale=35.0)
print('flow summary:', _flow_summary)

_fig, _axes = plt.subplots(1, 2, figsize=(18, 7))
_axes[0].imshow(as_rgb(_curr_frame)); _axes[0].set_title(f'Frame {_demo_idx}'); _axes[0].axis('off')
_axes[1].imshow(as_rgb(_flow_overlay)); _axes[1].set_title(f'Optical flow overlay (ROI {_roi})'); _axes[1].axis('off')
_fig.suptitle('Module 5: overlay motion view')
_fig.tight_layout()
_fig.savefig(LAB_DIR / 'module5_overlay_flow.png', dpi=130)
plt.show()



## Lab artifact export for the apps

Every module cell above has saved its figure to `output/lab-artifacts/`.
This final cell writes `manifest.json` so the web-dashboard and the Tauri
desktop app can list and render the workshop visualizations without running
the notebook.



In [ ]:
manifest = []
for artifact in sorted(LAB_DIR.glob('*.png')):
    manifest.append({
        'name': artifact.name,
        'path': str(artifact.relative_to(ROOT)),
        'module': artifact.name.split('_')[0],
    })
manifest_path = LAB_DIR / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('manifest:', manifest_path, '| artifacts:', len(manifest))
for entry in manifest:
    print(' -', entry['name'])



## What To Look For

Use this notebook to tune the production runner deliberately:
- If `frame_diff_score` stays low, increase OCR reuse aggressiveness before increasing model concurrency.
- If vehicle proposals are clean but OCR remains poor, the next gain is usually a dedicated plate detector or stronger OCR backend, not more rescans.
- If geospatial clustering is noisy, inspect `support_frames`, `geo_spread_m`, and `geolocation_mode` in the fused outputs before changing dashboard logic.

A practical follow-up is to duplicate the notebook and compare two parameter sets side by side, such as `ocr_frame_interval=3` versus `ocr_frame_interval=5`, or reuse enabled versus disabled.